# README: Predictive Model Development & Validation

This notebook presents a complete, production-oriented machine learning workflow for predicting customer churn. The objective is to identify customers at risk of leaving and support data-driven retention strategies.

The workflow follows best practices including data preparation, feature engineering, model development, hyperparameter tuning, evaluation, and risk analysis.

Key components:
- Business understanding and target definition
- Data preparation and feature engineering
- Model training and comparison
- Performance validation
- Interpretation, risks, and monitoring


## 1. Business Need & Target Definition

The objective is to predict customer churn (whether a customer will leave the company).

- **Target Variable:** Churn (Yes/No)
- **Unit of Analysis:** Individual customer
- **Prediction Horizon:** Next billing cycle
- **Success Criteria:** ROC-AUC > 0.75 and Recall improvement over baseline


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve


## 2. Data Preparation & Feature Engineering

In [ ]:
# Load dataset
data = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Drop ID
data.drop('customerID', axis=1, inplace=True)

# Encode target
data['Churn'] = data['Churn'].map({'Yes':1, 'No':0})

# One-hot encoding
data = pd.get_dummies(data, drop_first=True)

# Split features and target
X = data.drop('Churn', axis=1)
y = data['Churn']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Baseline Model

In [ ]:
# Baseline: majority class
baseline_pred = np.zeros(len(y_test))
print('Baseline Accuracy:', (baseline_pred == y_test).mean())

## 4. Model Development

In [ ]:
# Logistic Regression with tuning
log_model = GridSearchCV(
    LogisticRegression(max_iter=1000),
    {'C':[0.01,0.1,1,10]},
    cv=5
)
log_model.fit(X_train_scaled, y_train)

# Random Forest with tuning
rf_model = GridSearchCV(
    RandomForestClassifier(),
    {'n_estimators':[100,200], 'max_depth':[5,10]},
    cv=5
)
rf_model.fit(X_train, y_train)

## 5. Evaluation

In [ ]:
# Predictions
log_preds = log_model.predict(X_test_scaled)
rf_preds = rf_model.predict(X_test)

# Reports
print('Logistic Regression\n', classification_report(y_test, log_preds))
print('Random Forest\n', classification_report(y_test, rf_preds))

# ROC-AUC
print('Logistic ROC-AUC:', roc_auc_score(y_test, log_model.predict_proba(X_test_scaled)[:,1]))
print('RF ROC-AUC:', roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1]))

In [ ]:
# Confusion Matrix
sns.heatmap(confusion_matrix(y_test, rf_preds), annot=True, fmt='d')
plt.title('Random Forest Confusion Matrix')
plt.show()

## 6. Interpretation, Risks & Monitoring

### Interpretation
Random Forest model identifies key drivers of churn such as contract type, tenure, and monthly charges.

### Risks
- Data leakage avoided by splitting before scaling
- Potential bias in demographic variables
- Model drift over time

### Monitoring
- Track ROC-AUC over time
- Monitor input data distribution
- Set alert thresholds for performance drop
